# Project: Data Cleaning, Analysis, and Visualization  
## Dataset: California Housing Prices

# Project Overview

In this project, we clean, explore, analyze, and visualize the California Housing dataset to uncover patterns that influence house prices.

The dataset contains housing information such as:
- Median income
- Total rooms
- Total bedrooms
- Population
- Households
- Latitude & longitude
- Ocean proximity
- Median house value


# PART 1: Data Cleaning Objectives


### 1️ Data Inspection
- Display the first 5 rows
- Check dataset shape
- Identify data types
- Generate summary statistics
- Identify missing values

### 2️ Missing Value Treatment
- Identify missing values in `total_bedrooms`
- Compare:
  - Mean imputation
  - Median imputation
- Justify which method is better

### 3️ Outlier Detection
- Detect outliers in:
  - `median_house_value`
  - `median_income`
- Use:
  - Boxplots
  - IQR method
- Decide whether to:
  - Cap outliers
  - Remove them
  - Keep them (with justification)

### 4️ Feature Engineering
Create new features:
- `rooms_per_household`
- `bedrooms_per_room`
- `population_per_household`

Explain why each feature may improve analysis.



#  PART 2: Exploratory Data Analysis (EDA)

Answer the following questions using pandas:

### 1️ Which variables are most correlated with house value?
- Produce a correlation matrix
- Identify top 3 positive correlations
- Identify top negative correlation

### 2️ Does income strongly influence house prices?
- Calculate correlation between `median_income` and `median_house_value`
- Interpret the strength (weak/moderate/strong)

### 3️ Do coastal houses cost more?
- Group by `ocean_proximity`
- Compare average `median_house_value`
- Rank categories from highest to lowest price

### 4️ Does household density affect pricing?
- Analyze `population_per_household`
- Compare against house value


# PART 3: Visualization Objectives

Create and interpret the following visualizations:

### 1️ Distribution Plots
- Histogram of `median_house_value`
- Histogram of `median_income`
- Interpret skewness

### 2️ Boxplots
- House value by ocean proximity
- Detect spread and outliers

### 3️ Scatter Plots
- Income vs House Value
- Rooms per household vs House Value
- Longitude vs Latitude (colored by house value)

### 4️ Heatmap
- Correlation heatmap
- Identify strongest predictors visually


# PART 4: Insight & Business Interpretation

Students must write a short report(conclusion) answering:

1. What is the strongest predictor of house price?
2. Are coastal properties significantly more expensive?
3. Does number of rooms directly increase house value?
4. If you were a real estate investor, what variable would you prioritize?



# Deliverables

Students must submit:

- Cleaned dataset
- Jupyter Notebook
- At least 6 visualizations

## Data Source 

In [13]:
# No download required 
# dataframe is df_califonia
# After  loading the
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)

df_califonia = data.frame


In [ ]:
# oops this dataset is already clean lets add impurities
# Run this after  loading data (both from file or sklearn)

# set seed so we have simmilar impurities 
np.random.seed(42)


#  Rename columns to match project questions

df_califonia.rename(columns={
    "MedInc": "median_income",
    "AveRooms": "total_rooms",
    "AveBedrms": "total_bedrooms",
    "Population": "population",
    "AveOccup": "households",
    "MedHouseVal": "median_house_value",
    "Latitude": "latitude",
    "Longitude": "longitude"
}, inplace=True)



#  Create Ocean Proximity (categorical)

conditions = [
    df_califonia["longitude"] < -122,
    df_califonia["longitude"].between(-122, -118),
    df_califonia["longitude"] > -118
]

choices = ["NEAR OCEAN", "<1H OCEAN", "INLAND"]

df_califonia["ocean_proximity"] = np.select(
    conditions,
    choices,
    default="ISLAND"
)


# Introduce inconsistent formatting
df_califonia.loc[df_califonia.sample(40).index, "ocean_proximity"] = "near ocean"



# Add Missing Values total_bedrooms


df_califonia.loc[
    df_califonia.sample(frac=0.1).index,
    "total_bedrooms"
] = np.nan

# Add some missing median_income
df_califonia.loc[
    df_califonia.sample(frac=0.05).index,
    "median_income"
] = np.nan

# Add some missing median_income
df_califonia.loc[
    df_califonia.sample(frac=0.005).index,
    "ocean_proximity"
] = np.nan


#  Add Duplicates
df_califonia = pd.concat(
    [df_califonia, df_califonia.sample(50)],
    ignore_index=True
)


# Add Extreme Outliers

# Inflate income
df_califonia.loc[
    df_califonia.sample(15).index,
    "median_income"
] *= 20

# Inflate house values
df_califonia.loc[
    df_califonia.sample(15).index,
    "median_house_value"
] *= 8

# Add unrealistic room counts
df_califonia.loc[
    df_califonia.sample(10).index,
    "total_rooms"
] *= 25


# Create Slight Skew


df_califonia["population"] = (
    df_califonia["population"] ** 1.3
)


# Shuffle dataset

df_califonia = df_califonia.sample(frac=1).reset_index(drop=True)


In [15]:
df_califonia.info()

<class 'pandas.DataFrame'>
RangeIndex: 20690 entries, 0 to 20689
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   median_income       19655 non-null  float64
 1   HouseAge            20690 non-null  float64
 2   total_rooms         20690 non-null  float64
 3   total_bedrooms      18618 non-null  float64
 4   population          20690 non-null  float64
 5   households          20690 non-null  float64
 6   latitude            20690 non-null  float64
 7   longitude           20690 non-null  float64
 8   median_house_value  20690 non-null  float64
 9   ocean_proximity     20587 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [16]:
df_califonia.head()

,median_income,HouseAge,total_rooms,total_bedrooms,population,households,latitude,longitude,median_house_value,ocean_proximity
0,2.0437,31.0,4.047817,1.057173,41842.996989,3.731809,33.93,-118.29,1.37200,<1H OCEAN
1,5.8519,36.0,6.771812,1.100671,2943.755200,3.127517,34.07,-117.90,2.49400,INLAND
2,3.4801,52.0,3.977155,1.185877,11283.729965,1.360332,37.80,-122.44,5.00001,near ocean
3,4.1319,33.0,5.262530,1.081146,10528.295711,2.964200,34.19,-118.54,2.03700,<1H OCEAN
4,2.5446,27.0,22.646341,3.953659,9364.716189,2.768293,34.26,-117.16,1.35200,INLAND


In [17]:
df_califonia.describe()

,median_income,HouseAge,total_rooms,total_bedrooms,population,households,latitude,longitude,median_house_value
count,19655.000000,20690.000000,20690.000000,18618.000000,20690.000000,20690.000000,20690.000000,20690.000000,20690.000000
mean,3.913946,28.644901,5.485834,1.096534,13770.318325,3.069864,35.633329,-119.571306,2.079112
std,2.637819,12.583241,3.573139,0.475832,16896.238977,10.373543,2.136288,2.004293,1.235815
min,0.499900,1.000000,0.846154,0.333333,4.171168,0.692308,32.540000,-124.350000,0.149990
25%,2.564150,18.000000,4.441920,1.005812,5808.303613,2.429379,33.930000,-121.800000,1.196000
50%,3.533500,29.000000,5.230769,1.048421,9687.767399,2.818016,34.260000,-118.500000,1.798000
75%,4.743400,37.000000,6.054487,1.099429,16124.952350,3.281522,37.710000,-118.010000,2.650000
max,120.848000,52.000000,158.093525,34.066667,828292.913119,1243.333333,41.950000,-114.310000,32.000000
